In [1]:
import boto3
import os
import json
bedrock = boto3.client('bedrock', region_name='us-east-2')
response = bedrock.list_foundation_models()
print(f"Available models: {len(response['modelSummaries'])}")
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-2')

Available models: 96


In [2]:
class Conversation:
    """Multi-turn chat that keeps the message history and re-sends it on every call.

    - Token limit: history is capped at `max_history_tokens` (estimated, ~4 chars/token).
      When exceeded, older turns are summarized (auto_summarize=True) or simply dropped.
    - Summary: older turns are folded into `self.summary`, which is sent in the system prompt.
    - Persistence: save() / load() store everything in a JSON file.
    - chat(): interactive terminal loop.
    """

    def __init__(self, system=None, model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0',
                 max_tokens=500, max_history_tokens=2000, keep_recent=4, auto_summarize=True):
        self.system = system
        self.model_id = model_id
        self.max_tokens = max_tokens
        self.max_history_tokens = max_history_tokens
        self.keep_recent = keep_recent  # messages kept verbatim when summarizing
        self.auto_summarize = auto_summarize
        self.messages = []  # [{"role": "user" | "assistant", "content": str}]
        self.summary = ""

    # ---- Token accounting -------------------------------------------------
    @staticmethod
    def estimate_tokens(text):
        return len(text) // 4 + 1

    def history_tokens(self):
        total = sum(self.estimate_tokens(m["content"]) for m in self.messages)
        return total + (self.estimate_tokens(self.summary) if self.summary else 0)

    # ---- Model call -------------------------------------------------------
    def _invoke(self, messages, system=None, max_tokens=None):
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens or self.max_tokens,
            "messages": messages,
        }
        if system:
            body["system"] = system
        response = bedrock_runtime.invoke_model(modelId=self.model_id, body=json.dumps(body))
        return json.loads(response["body"].read())["content"][0]["text"]

    def _system_prompt(self):
        parts = [self.system] if self.system else []
        if self.summary:
            parts.append(f"Summary of the earlier part of this conversation:\n{self.summary}")
        return "\n\n".join(parts) or None

    # ---- Summarization / token limit --------------------------------------
    def summarize(self, keep_recent=None):
        """Fold older messages into self.summary, keeping the last `keep_recent` verbatim."""
        keep = self.keep_recent if keep_recent is None else keep_recent
        cut = max(len(self.messages) - keep, 0)
        while cut < len(self.messages) and self.messages[cut]["role"] != "user":
            cut += 1  # the remaining history must start with a user message
        older = self.messages[:cut]
        if not older:
            return self.summary
        transcript = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in older)
        prompt = ("Summarize this conversation concisely, preserving facts, names, decisions and "
                  "open questions the assistant needs to continue it.\n\n")
        if self.summary:
            prompt += f"Previous summary:\n{self.summary}\n\n"
        prompt += f"New messages:\n{transcript}"
        self.summary = self._invoke([{"role": "user", "content": prompt}], max_tokens=400).strip()
        self.messages = self.messages[cut:]
        return self.summary

    def _enforce_limit(self):
        if self.history_tokens() <= self.max_history_tokens:
            return
        if self.auto_summarize and len(self.messages) > self.keep_recent:
            self.summarize()
        # Still too long (or summarizing disabled): drop the oldest turns
        while self.history_tokens() > self.max_history_tokens and len(self.messages) > 1:
            self.messages.pop(0)
            while self.messages and self.messages[0]["role"] != "user":
                self.messages.pop(0)

    # ---- Chat -------------------------------------------------------------
    def send(self, user_text):
        """Add a user message, call the model with the full history and store the reply."""
        self.messages.append({"role": "user", "content": user_text})
        try:
            self._enforce_limit()
            reply = self._invoke(self.messages, system=self._system_prompt())
        except Exception:
            # Keep history consistent: remove the unanswered user message
            if self.messages and self.messages[-1]["content"] == user_text:
                self.messages.pop()
            raise
        self.messages.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.messages, self.summary = [], ""

    # ---- Save / load ------------------------------------------------------
    def save(self, path):
        data = {
            "system": self.system, "model_id": self.model_id, "max_tokens": self.max_tokens,
            "max_history_tokens": self.max_history_tokens, "keep_recent": self.keep_recent,
            "auto_summarize": self.auto_summarize, "summary": self.summary,
            "messages": self.messages,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Conversation saved to {os.path.abspath(path)} ({len(self.messages)} messages)")

    @classmethod
    def load(cls, path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        convo = cls(system=data.get("system"), model_id=data["model_id"],
                    max_tokens=data["max_tokens"], max_history_tokens=data["max_history_tokens"],
                    keep_recent=data["keep_recent"], auto_summarize=data["auto_summarize"])
        convo.summary = data.get("summary", "")
        convo.messages = data["messages"]
        return convo

    # ---- Interactive terminal chatbot -------------------------------------
    def chat(self, save_path="conversation.json"):
        """Interactive loop. Commands: /save [file], /load [file], /summary, /tokens, /reset, /quit"""
        print("Chatbot ready. Commands: /save [file] /load [file] /summary /tokens /reset /quit")
        while True:
            try:
                user_text = input("\nYou: ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\nBye!")
                break
            if not user_text:
                continue
            command, _, arg = user_text.partition(" ")
            if command == "/quit":
                print("Bye!")
                break
            elif command == "/save":
                self.save(arg or save_path)
            elif command == "/load":
                try:
                    loaded = Conversation.load(arg or save_path)
                except (OSError, ValueError, KeyError) as e:
                    print(f"Could not load: {e}")
                    continue
                self.__dict__.update(loaded.__dict__)
                print(f"Loaded {len(self.messages)} messages.")
            elif command == "/summary":
                print(self.summarize(keep_recent=0) or "(nothing to summarize)")
            elif command == "/tokens":
                print(f"~{self.history_tokens()}/{self.max_history_tokens} history tokens, "
                      f"{len(self.messages)} messages")
            elif command == "/reset":
                self.reset()
                print("History cleared.")
            else:
                try:
                    print(f"User: {user_text}")
                    print(f"Claude: {self.send(user_text)}")
                except (ClientError, BotoCoreError) as e:
                    print(f"[Error: {e}]")

In [3]:
# 1. Context is remembered across turns
convo = Conversation(system="You are a concise assistant.")
print(convo.send("Hi! My name is Matias and I'm learning AWS Bedrock."))
print(convo.send("What's my name and what am I learning?"))

# 2. Token limit + automatic summarization (tiny limit so it triggers quickly)
short = Conversation(max_history_tokens=250, keep_recent=2)
for question in ["Tell me about Mexico City in 2 sentences.",
                 "And about Oaxaca, 2 sentences.",
                 "And about Monterrey, 2 sentences.",
                 "Which of those cities did I ask about first?"]:
    print(f"\nYou: {question}\nClaude: {short.send(question)}")
    print(f"  [~{short.history_tokens()}/{short.max_history_tokens} tokens, {len(short.messages)} messages]")
print("\nSummary kept:", short.summary)

# 3. Save / load
short.save("conversation.json")
restored = Conversation.load("conversation.json")
print(restored.send("Remind me what we talked about."))

Hello Matias! Great to meet you. AWS Bedrock is an excellent service to learn - it provides easy access to foundation models from various AI companies through a unified API.

What aspect of Bedrock are you most interested in exploring? Model selection, API integration, RAG (Retrieval Augmented Generation), or something else?
Your name is **Matias** and you're learning **AWS Bedrock**.

You: Tell me about Mexico City in 2 sentences.
Claude: Mexico City is the capital and largest city of Mexico, with over 9 million residents in the city proper and more than 21 million in the greater metropolitan area, making it one of the most populous urban centers in the world. Built on the ruins of the ancient Aztec capital of Tenochtitlan, it sits at high altitude in the Valley of Mexico and is known for its rich history, vibrant culture, world-class museums, and diverse cuisine.
  [~120/250 tokens, 2 messages]

You: And about Oaxaca, 2 sentences.
Claude: Oaxaca is a state in southern Mexico renowned

In [4]:
# 4. Interactive chatbot (type /quit to exit; /save, /load, /summary, /tokens, /reset also available)
Conversation(system="You are a friendly assistant.").chat()

Chatbot ready. Commands: /save [file] /load [file] /summary /tokens /reset /quit
User: Hi
Claude: Hello! 👋 How can I help you today?
User: How are you feeling today?
Claude: I'm doing well, thank you for asking! I'm here and ready to help with whatever you need. 

I don't experience feelings quite the way humans do, but I appreciate the friendly question. How are *you* doing today? Is there anything I can help you with?
User: Ok, bye
Claude: Goodbye! Feel free to come back anytime if you need help with anything. Have a great day! 👋
Bye!
